# GuidaPlate — XGBoost v3 (Raw Features, Clinical Labels)
## Notebook 04c — Leakage-reduced classifier

**Does not modify** notebooks 04/04b or `models/xgboost_v1.pkl`.

Uses v3 clinical-score labels and **raw nutrient features only** (no ratio features).


In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, f1_score, roc_curve, auc,
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_sample_weight
from statsmodels.stats.contingency_tables import mcnemar
import xgboost as xgb

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8')
except OSError:
    plt.style.use('seaborn')
%matplotlib inline

RANDOM_STATE = 42
TEST_SIZE = 0.2


def project_root() -> Path:
    p = Path.cwd().resolve()
    if p.name == 'notebooks':
        return p.parent
    if (p / 'data' / 'processed' / 'ckd_cohort_final.csv').exists():
        return p
    if (p.parent / 'data' / 'processed' / 'ckd_cohort_final.csv').exists():
        return p.parent
    return p

ROOT = project_root()
FIG_DIR = ROOT / 'outputs' / 'figures'
STATS_DIR = ROOT / 'outputs' / 'stats'
MODEL_DIR = ROOT / 'models'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(STATS_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

KDOQI = {
    'G2':  {'potassium': 3500, 'phosphorus': 1000, 'protein_per_kg': 0.8, 'sodium': 2300},
    'G3a': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G3b': {'potassium': 3000, 'phosphorus': 800,  'protein_per_kg': 0.6, 'sodium': 2300},
    'G4':  {'potassium': 2500, 'phosphorus': 700,  'protein_per_kg': 0.55, 'sodium': 2300},
}

WEIGHTS = {
    'potassium': 0.35,
    'phosphorus': 0.30,
    'protein_per_kg': 0.25,
    'sodium': 0.10,
}

RISK_CLASSES = ['LOW', 'MODERATE', 'HIGH']
STAGE_ENCODE = {'G2': 1, 'G3a': 2, 'G3b': 3, 'G4': 4}
RISK_ENCODE = {c: i for i, c in enumerate(RISK_CLASSES)}

def compute_clinical_score(row):
    stage = row['ckd_stage']
    if stage not in KDOQI:
        return np.nan
    limits = KDOQI[stage]
    score = 0.0
    for nutrient, weight in WEIGHTS.items():
        if pd.isna(row.get(nutrient)):
            continue
        ratio = row[nutrient] / limits[nutrient]
        if ratio > 1.0:
            score += weight * (1 + (ratio - 1) * 2)
        else:
            score += weight * ratio
    return score

def assign_clinical_risk_label(row):
    score = compute_clinical_score(row)
    if pd.isna(score):
        return None
    if score >= 1.2:
        return 'HIGH'
    if score >= 0.7:
        return 'MODERATE'
    return 'LOW'

def assign_rule_baseline_label(row):
    stage = row['ckd_stage']
    if stage not in KDOQI:
        return None
    limits = KDOQI[stage]
    exceeded = 0
    for nutrient in ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']:
        if pd.notna(row.get(nutrient)) and row[nutrient] > limits[nutrient]:
            exceeded += 1
    if exceeded >= 2:
        return 'HIGH'
    if exceeded == 1:
        return 'MODERATE'
    return 'LOW'

print(f'Project root: {ROOT}')


## Section 1 — Setup


In [ ]:
cohort = pd.read_csv(ROOT / 'data' / 'processed' / 'ckd_cohort_final.csv')
labels_v3 = pd.read_csv(STATS_DIR / '05_risk_labels_v3.csv')

df = cohort.merge(labels_v3, on='SEQN', how='inner', suffixes=('', '_v3'))
df = df.dropna(subset=['risk_label', 'clinical_score'])
nutrient_cols = ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']
df = df.dropna(subset=nutrient_cols)

print(f'Shape: {df.shape}')
print('v3 label distribution:')
print(df['risk_label'].value_counts().reindex(RISK_CLASSES))


## Section 2 — Feature Construction


In [ ]:
df['ckd_stage_encoded'] = df['ckd_stage'].map(STAGE_ENCODE)
df['stage_numeric'] = df['ckd_stage'].map({'G2': 2, 'G3a': 3, 'G3b': 3, 'G4': 4})
df['k_p_product'] = (df['potassium'] * df['phosphorus']) / 1e6
df['protein_sodium_ratio'] = df['protein_per_kg'] / (df['sodium'] / 1000 + 1e-6)
# clinical_score already loaded from v3 labels file

FEATURES_V3 = [
    'potassium', 'phosphorus', 'protein_per_kg', 'sodium',
    'ckd_stage_encoded', 'stage_numeric', 'k_p_product',
    'protein_sodium_ratio', 'clinical_score',
]

LEAKAGE_FEATURES = [
    'potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio',
]
assert not any(f in FEATURES_V3 for f in LEAKAGE_FEATURES), 'Ratio leakage features must not be included'

df['risk_encoded'] = df['risk_label'].map(RISK_ENCODE)
X = df[FEATURES_V3]
y = df['risk_encoded']

print('v3 feature list (no ratio features):')
for f in FEATURES_V3:
    print(f'  - {f}')


In [ ]:
# DIAGRAM 5 — Quick untuned XGB feature importance baseline
quick = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3, n_estimators=100,
    max_depth=4, random_state=RANDOM_STATE, eval_metric='mlogloss', verbosity=0,
)
quick.fit(X, y)

imp = pd.Series(quick.feature_importances_, index=FEATURES_V3).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind='barh', ax=ax, color='#0f766e')
ax.set_title('Baseline Feature Importance (Untuned XGBoost v3)')
ax.set_xlabel('Importance (weight/gain)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_05_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_05_feature_importance.png"}')
print('Confirmed: no ratio features in model.')


## Section 3 — Train/Test Split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)
test_idx = X_test.index
X_train_df = pd.DataFrame(X_train, columns=FEATURES_V3)
X_test_df = pd.DataFrame(X_test, columns=FEATURES_V3)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print('Train class distribution:')
print(y_train.value_counts().sort_index())
print('Test class distribution:')
print(y_test.value_counts().sort_index())


## Section 4 — Cost-sensitive Training


In [ ]:
class_weight = {RISK_ENCODE['HIGH']: 1.0, RISK_ENCODE['MODERATE']: 4.0, RISK_ENCODE['LOW']: 1.0}
sample_weight_train = compute_sample_weight(class_weight=class_weight, y=y_train)

param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.1, 0.5, 1.0],
    'reg_lambda': [1.0, 1.5, 2.0],
}

base_model = xgb.XGBClassifier(
    objective='multi:softprob', num_class=3,
    random_state=RANDOM_STATE, eval_metric='mlogloss',
)

search = RandomizedSearchCV(
    base_model, param_distributions=param_distributions,
    n_iter=50, scoring='f1_macro',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1,
)

print('Starting RandomizedSearchCV (50 iter, f1_macro)...')
t0 = time.time()
search.fit(X_train, y_train, sample_weight=sample_weight_train)
print(f'Done in {time.time()-t0:.1f}s | Best CV F1 macro: {search.best_score_:.4f}')
print('Best params:', search.best_params_)

best_model = xgb.XGBClassifier(
    **search.best_params_, objective='multi:softprob', num_class=3,
    eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0,
)
best_model.fit(X_train, y_train, sample_weight=sample_weight_train,
               eval_set=[(X_test, y_test)], verbose=False)


## Section 5 — Full Evaluation


In [ ]:
y_pred_v3 = best_model.predict(X_test_df)
y_prob_v3 = best_model.predict_proba(X_test_df)

# Load v1/v2 for comparison (evaluated against v3 test labels)
v1_model = joblib.load(MODEL_DIR / 'xgboost_v1.pkl')
v2_path = MODEL_DIR / 'xgboost_v2.pkl'
v2_model = joblib.load(v2_path) if v2_path.exists() else None

FEATURES_V1 = [
    'potassium', 'phosphorus', 'protein_per_kg', 'sodium',
    'potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio',
    'ckd_stage_encoded',
]

def add_v1_ratios(frame):
    out = frame.copy()
    for nutrient in ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']:
        out[f'{nutrient.split("_")[0] if nutrient != "protein_per_kg" else "protein"}_ratio'] = np.nan
    out['potassium_ratio'] = out.apply(lambda r: r['potassium']/KDOQI[r['ckd_stage']]['potassium'] if r['ckd_stage'] in KDOQI else np.nan, axis=1)
    out['phosphorus_ratio'] = out.apply(lambda r: r['phosphorus']/KDOQI[r['ckd_stage']]['phosphorus'] if r['ckd_stage'] in KDOQI else np.nan, axis=1)
    out['protein_ratio'] = out.apply(lambda r: r['protein_per_kg']/KDOQI[r['ckd_stage']]['protein_per_kg'] if r['ckd_stage'] in KDOQI else np.nan, axis=1)
    out['sodium_ratio'] = out.apply(lambda r: r['sodium']/KDOQI[r['ckd_stage']]['sodium'] if r['ckd_stage'] in KDOQI else np.nan, axis=1)
    return out

test_rows = df.loc[test_idx]
test_v1 = add_v1_ratios(test_rows)
y_pred_v1 = v1_model.predict(test_v1[FEATURES_V1])

if v2_model is not None:
    # v2 used extended features — rebuild for test rows only if needed
    pass

def class_recall(y_true, y_pred, cls):
    idx = RISK_ENCODE[cls]
    tp = np.sum((y_true == idx) & (y_pred == idx))
    fn = np.sum((y_true == idx) & (y_pred != idx))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0

def per_class_f1(y_true, y_pred, cls):
    idx = RISK_ENCODE[cls]
    return f1_score(y_true == idx, y_pred == idx, zero_division=0)

v3_acc = accuracy_score(y_test, y_pred_v3)
v3_f1_w = f1_score(y_test, y_pred_v3, average='weighted', zero_division=0)
v3_f1_m = f1_score(y_test, y_pred_v3, average='macro', zero_division=0)
v3_auc = roc_auc_score(y_test, y_prob_v3, multi_class='ovr', average='weighted')
v3_high_sens = class_recall(y_test, y_pred_v3, 'HIGH')
v3_mod_sens = class_recall(y_test, y_pred_v3, 'MODERATE')

v1_f1_m = f1_score(y_test, y_pred_v1, average='macro', zero_division=0)
v1_mod_sens = class_recall(y_test, y_pred_v1, 'MODERATE')

print(classification_report(y_test, y_pred_v3, target_names=RISK_CLASSES, zero_division=0))


In [ ]:
# DIAGRAM 6 — Confusion matrix v3
cm_labels = ['HIGH', 'LOW', 'MODERATE']
cm_v3 = confusion_matrix(y_test, y_pred_v3, labels=[RISK_ENCODE[c] for c in cm_labels])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_v3, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=cm_labels, yticklabels=cm_labels, linewidths=0.5)
ax.set_title('XGBoost v3 Confusion Matrix (v3 labels)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_06_confusion.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_06_confusion.png"}')

# DIAGRAM 7 — ROC curves
# Fix: y_prob columns are ordered LOW=0, MODERATE=1, HIGH=2 (RISK_ENCODE),
# while y_test_bin columns follow `classes` display order (HIGH, LOW, MODERATE).
# Index probabilities by RISK_ENCODE[cls], not by loop index i.
classes = ['HIGH', 'LOW', 'MODERATE']
y_test_bin = label_binarize(y_test, classes=[RISK_ENCODE[c] for c in classes])
fig, ax = plt.subplots(figsize=(8, 6))
colors_roc = ['#ef4444', '#22c55e', '#f59e0b']
for i, (cls, color) in enumerate(zip(classes, colors_roc)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_v3[:, RISK_ENCODE[cls]])
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{cls} (AUC = {auc(fpr, tpr):.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
_weighted_auc = roc_auc_score(y_test, y_prob_v3, multi_class='ovr', average='weighted')
ax.set_title(f'XGBoost v3 ROC Curves by Class (weighted AUC = {_weighted_auc:.3f})')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_07_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_07_roc.png"}')


In [ ]:
# DIAGRAM 8 — Learning curves
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=5, scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1,
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#0f766e', label='Training')
ax.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
                train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1, color='#0f766e')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#f59e0b', label='Validation')
ax.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
                val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.1, color='#f59e0b')
ax.set_xlabel('Training Set Size'); ax.set_ylabel('F1 Macro')
ax.set_title('XGBoost v3 Learning Curves'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_08_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_08_learning_curves.png"}')


In [ ]:
# DIAGRAM 9 — Per-class F1: v1 vs v2 vs v3 (ground truth = v3 labels)
v1_high_f1 = per_class_f1(y_test, y_pred_v1, 'HIGH')
v1_low_f1 = per_class_f1(y_test, y_pred_v1, 'LOW')
v1_mod_f1 = per_class_f1(y_test, y_pred_v1, 'MODERATE')
v3_high_f1 = per_class_f1(y_test, y_pred_v3, 'HIGH')
v3_low_f1 = per_class_f1(y_test, y_pred_v3, 'LOW')
v3_mod_f1 = per_class_f1(y_test, y_pred_v3, 'MODERATE')

# v2 on same test set — rebuild v2 features for test rows
if v2_model is not None:
    tdf = test_rows.copy()
    for nutrient in ['potassium', 'phosphorus', 'protein_per_kg', 'sodium']:
        pass
    tdf = add_v1_ratios(tdf)
    tdf['k_x_p'] = tdf['potassium_ratio'] * tdf['phosphorus_ratio']
    tdf['k_x_protein'] = tdf['potassium_ratio'] * tdf['protein_ratio']
    tdf['p_x_protein'] = tdf['phosphorus_ratio'] * tdf['protein_ratio']
    ratio_cols = ['potassium_ratio', 'phosphorus_ratio', 'protein_ratio', 'sodium_ratio']
    tdf['total_burden'] = tdf[ratio_cols].sum(axis=1)
    tdf['max_ratio'] = tdf[ratio_cols].max(axis=1)
    tdf['nutrients_near_limit'] = tdf[ratio_cols].apply(lambda r: ((r >= 0.8) & (r < 1.0)).sum(), axis=1)
    tdf['nutrients_exceeded'] = tdf[ratio_cols].apply(lambda r: (r >= 1.0).sum(), axis=1)
    tdf['ckd_stage_encoded'] = tdf['ckd_stage'].map(STAGE_ENCODE)
    FEATURES_V2 = FEATURES_V1 + ['k_x_p','k_x_protein','p_x_protein','total_burden','max_ratio','nutrients_near_limit','nutrients_exceeded']
    y_pred_v2 = v2_model.predict(tdf[FEATURES_V2])
    v2_high_f1 = per_class_f1(y_test, y_pred_v2, 'HIGH')
    v2_low_f1 = per_class_f1(y_test, y_pred_v2, 'LOW')
    v2_mod_f1 = per_class_f1(y_test, y_pred_v2, 'MODERATE')
else:
    v2_high_f1 = v2_low_f1 = v2_mod_f1 = 0.0

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(3)
w = 0.25
ax.bar(x - w, [v1_high_f1, v1_low_f1, v1_mod_f1], w, label='v1 Original', color='#94a3b8')
ax.bar(x, [v2_high_f1, v2_low_f1, v2_mod_f1], w, label='v2 (leaked)', color='#f59e0b')
ax.bar(x + w, [v3_high_f1, v3_low_f1, v3_mod_f1], w, label='v3 Clinical', color='#0f766e')
ax.set_xticks(x); ax.set_xticklabels(['HIGH', 'LOW', 'MODERATE'])
ax.set_ylabel('F1 Score'); ax.set_title('Per-Class F1: v1 vs v2 vs v3 (v3 ground truth)')
ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_09_f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_09_f1_comparison.png"}')


In [ ]:
# DIAGRAM 10 & 11 — SHAP beeswarm HIGH and MODERATE
# Bug fix: shap.summary_plot(..., plot_type='beeswarm') with shap 0.51 +
# matplotlib 3.11 draws empty axes (0 PathCollections). Use native XGBoost
# pred_contribs + shap.plots.beeswarm(Explanation) instead — same as meal figures.
_dmat = xgb.DMatrix(X_test_df, feature_names=list(FEATURES_V3))
_contribs = np.asarray(best_model.get_booster().predict(_dmat, pred_contribs=True))
# shape (n, n_classes, n_features+1); bias column last
assert _contribs.ndim == 3 and _contribs.shape[2] == len(FEATURES_V3) + 1, _contribs.shape

for _cls_name, _out_name in (
    ('HIGH', 'xgb_v3_10_shap_high.png'),
    ('MODERATE', 'xgb_v3_11_shap_moderate.png'),
):
    _idx = RISK_ENCODE[_cls_name]
    _values = _contribs[:, _idx, : len(FEATURES_V3)]
    assert float(np.abs(_values).mean()) > 0, f'empty SHAP values for {_cls_name}'
    _explanation = shap.Explanation(
        values=_values,
        base_values=_contribs[:, _idx, len(FEATURES_V3)],
        data=X_test_df.to_numpy(),
        feature_names=list(FEATURES_V3),
    )
    plt.figure(figsize=(10, 7))
    shap.plots.beeswarm(_explanation, show=False, max_display=15)
    plt.title(f'SHAP Beeswarm — {_cls_name} Class (v3 raw features)', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIG_DIR / _out_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIG_DIR / _out_name} (mean|shap|={float(np.abs(_values).mean()):.4f})')


## Section 6 — McNemar Test


In [ ]:
test_df = df.loc[test_idx].copy()
y_true_v3 = y_test.values
y_rule = test_df.apply(assign_rule_baseline_label, axis=1).map(RISK_ENCODE).values

v3_correct = (y_pred_v3 == y_true_v3)
rule_correct = (y_rule == y_true_v3)
v1_correct = (y_pred_v1 == y_true_v3)

b_v3_rule = int(np.sum(rule_correct & ~v3_correct))
c_v3_rule = int(np.sum(v3_correct & ~rule_correct))
n00_v3 = int(np.sum(v3_correct & rule_correct))
n11_v3 = int(np.sum(~v3_correct & ~rule_correct))
if b_v3_rule + c_v3_rule > 0:
    v3_mcnemar_p = float(mcnemar(
        np.array([[n00_v3, c_v3_rule], [b_v3_rule, n11_v3]]), exact=True
    ).pvalue)
else:
    v3_mcnemar_p = 1.0

b_v1_v3 = int(np.sum(v1_correct & ~v3_correct))
c_v1_v3 = int(np.sum(v3_correct & ~v1_correct))
n00_v1 = int(np.sum(v3_correct & v1_correct))
n11_v1 = int(np.sum(~v3_correct & ~v1_correct))
if b_v1_v3 + c_v1_v3 > 0:
    v1_v3_mcnemar_p = float(mcnemar(
        np.array([[n00_v1, c_v1_v3], [b_v1_v3, n11_v1]]), exact=True
    ).pvalue)
else:
    v1_v3_mcnemar_p = 1.0

print('=' * 50)
print('McNEMAR TESTS (ground truth = v3 labels)')
print('=' * 50)
print(f'v3 vs rule baseline: b={b_v3_rule}, c={c_v3_rule}, p={v3_mcnemar_p:.4f}')
print(f'v3 vs v1:            b={b_v1_v3}, c={c_v1_v3}, p={v1_v3_mcnemar_p:.4f}')
print('Reference: v1 vs baseline p=0.50 | v2 vs baseline p=1.00')

# DIAGRAM 12 — McNemar p-value summary
fig, ax = plt.subplots(figsize=(8, 5))
comparisons = ['v1 vs baseline', 'v2 vs baseline', 'v3 vs baseline']
p_values = [0.50, 1.00, v3_mcnemar_p]
colors = ['#94a3b8', '#f59e0b', '#0f766e' if v3_mcnemar_p < 0.05 else '#ef4444']
bars = ax.bar(comparisons, p_values, color=colors)
ax.axhline(0.05, color='red', linestyle='--', label='p=0.05 threshold')
ax.set_ylabel('McNemar p-value')
ax.set_title('McNemar Test Results Summary')
ax.set_ylim(0, max(1.05, max(p_values) * 1.1))
for bar, p in zip(bars, p_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{p:.3f}', ha='center')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'xgb_v3_12_mcnemar.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "xgb_v3_12_mcnemar.png"}')


## Section 7 — Final Comparison + Decision Gate


In [ ]:
# Reference metrics from prior runs
V1_REF = {'accuracy': 0.9932, 'f1_weighted': 0.9933, 'f1_macro': v1_f1_m,
          'auc': 1.0, 'high_sensitivity': 0.9951, 'mod_sensitivity': v1_mod_sens,
          'mcnemar_p': 0.50, 'leakage': 'YES'}
V2_REF = {'accuracy': 1.0, 'f1_weighted': 1.0, 'f1_macro': 1.0,
          'auc': 1.0, 'high_sensitivity': 1.0, 'mod_sensitivity': 1.0,
          'mcnemar_p': 1.00, 'leakage': 'WORSE'}

leakage_resolved = v3_mcnemar_p < 0.05 and v3_acc < 0.99

print('=' * 72)
print('FINAL COMPARISON TABLE')
print('=' * 72)
print(f"{'Metric':<14} | {'v1 Original':<12} | {'v2 (leaked)':<12} | {'v3 Clinical':<12}")
print('-' * 72)
print(f"{'Accuracy':<14} | {V1_REF['accuracy']*100:>10.2f}% | {V2_REF['accuracy']*100:>10.2f}% | {v3_acc*100:>10.2f}%")
print(f"{'F1 weighted':<14} | {V1_REF['f1_weighted']:>12.4f} | {V2_REF['f1_weighted']:>12.4f} | {v3_f1_w:>12.4f}")
print(f"{'F1 macro':<14} | {V1_REF['f1_macro']:>12.4f} | {V2_REF['f1_macro']:>12.4f} | {v3_f1_m:>12.4f}")
print(f"{'AUC':<14} | {V1_REF['auc']:>12.4f} | {V2_REF['auc']:>12.4f} | {v3_auc:>12.4f}")
print(f"{'HIGH sens':<14} | {V1_REF['high_sensitivity']*100:>10.2f}% | {V2_REF['high_sensitivity']*100:>10.2f}% | {v3_high_sens*100:>10.2f}%")
print(f"{'MOD sens':<14} | {V1_REF['mod_sensitivity']*100:>10.2f}% | {V2_REF['mod_sensitivity']*100:>10.2f}% | {v3_mod_sens*100:>10.2f}%")
print(f"{'McNemar p':<14} | {V1_REF['mcnemar_p']:>12.2f} | {V2_REF['mcnemar_p']:>12.2f} | {v3_mcnemar_p:>12.4f}")
print(f"{'Leakage?':<14} | {V1_REF['leakage']:>12} | {V2_REF['leakage']:>12} | {'NO' if leakage_resolved else 'PARTIAL'}")
print('=' * 72)

v3_success = (
    v3_mcnemar_p < 0.05 and
    v3_acc >= 0.75 and
    v3_high_sens >= 0.80
)
v3_trade_off = (
    v3_mcnemar_p < 0.05 and
    v3_acc >= 0.65 and
    v3_mod_sens > 0.30
)

if v3_success:
    joblib.dump(best_model, MODEL_DIR / 'xgboost_v3.pkl')
    decision = '✅ V3 SUCCESS — leakage resolved'
    deploy_msg = 'Recommend deploying v3 to production'
elif v3_trade_off:
    joblib.dump(best_model, MODEL_DIR / 'xgboost_v3.pkl')
    decision = '⚠ V3 TRADE-OFF — leakage reduced'
    deploy_msg = 'Review before deploying'
else:
    decision = '❌ V3 did not resolve leakage'
    deploy_msg = 'Keep xgboost_v1.pkl in production'

print(decision)
print(f'McNemar p={v3_mcnemar_p:.4f}')
print(deploy_msg)

metrics_out = pd.DataFrame([{
    'model': 'XGBoost v3',
    'accuracy': round(v3_acc, 4),
    'f1_weighted': round(v3_f1_w, 4),
    'f1_macro': round(v3_f1_m, 4),
    'auc_roc': round(v3_auc, 4),
    'high_sensitivity': round(v3_high_sens, 4),
    'mod_sensitivity': round(v3_mod_sens, 4),
    'mcnemar_p': round(v3_mcnemar_p, 4),
    'mcnemar_b': b_v3_rule,
    'mcnemar_c': c_v3_rule,
    'decision': decision,
    'leakage_resolved': leakage_resolved,
    'n_features': len(FEATURES_V3),
    'training_samples': len(X_train),
    'test_samples': len(X_test),
}])
metrics_path = STATS_DIR / '10_xgboost_v3_metrics.csv'
metrics_out.to_csv(metrics_path, index=False)
print(f'\nSaved: {metrics_path}')
print('NOTEBOOK 04c COMPLETE')
